In [2]:
# ============================================================
# Meteorage — LightGBM v2
# Features enrichies + heuristiques physiques
# ============================================================

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    brier_score_loss, confusion_matrix,
)
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
HORIZON_MIN      = 30   # minutes : fenêtre de danger
INSIDE_RADIUS_KM = 20   # km : rayon opérationnel
TRAIN_RATIO      = 0.80 # fraction temporelle pour le train

# ============================================================
# 1. CHARGEMENT & NETTOYAGE
# ============================================================
df = pd.read_csv("../data_train_databattle2026/segment_alerts_all_airports_train.csv")

df["date"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values(["airport", "date"]).reset_index(drop=True)

# Nettoyage booléens
for col in ["icloud", "is_last_lightning_cloud_ground"]:
    if col in df.columns:
        df[col] = (
            df[col].astype(str).str.lower()
            .map({"true": 1, "false": 0, "1": 1, "0": 0, "nan": np.nan})
            .astype(float)
        )

# Colonnes numériques
for c in ["dist", "amplitude", "azimuth", "maxis"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Zone opérationnelle
df["inside20"] = (df["dist"] <= INSIDE_RADIUS_KM).astype(int)

print(f"Dataset chargé : {len(df)} lignes, {df['airport'].nunique()} aéroports")
print(f"Période : {df['date'].min()} → {df['date'].max()}")

# ============================================================
# 2. HELPERS
# ============================================================
def slope_from_series(y):
    """Pente linéaire sur positions équispacées."""
    y = pd.Series(y).dropna().values
    n = len(y)
    if n < 2:
        return np.nan
    x = np.arange(n)
    x_mean, y_mean = x.mean(), y.mean()
    denom = np.sum((x - x_mean) ** 2)
    if denom == 0:
        return np.nan
    return float(np.sum((x - x_mean) * (y - y_mean)) / denom)

def circular_std_deg(angles_deg):
    """Écart-type circulaire pour l'azimuth (degrés)."""
    a = pd.Series(angles_deg).dropna().values
    if len(a) == 0:
        return np.nan
    radians = np.deg2rad(a)
    C, S = np.mean(np.cos(radians)), np.mean(np.sin(radians))
    R = np.sqrt(C**2 + S**2)
    if R <= 1e-12:
        return 180.0
    return float(np.rad2deg(np.sqrt(-2 * np.log(R))))

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
def add_features(g):
    g = g.sort_values("date").copy()

    # ----------------------------------------------------------
    # GROUPE 1 — GAPS (rythme des éclairs)
    # ----------------------------------------------------------
    g["gap_min"]    = g["date"].diff().dt.total_seconds() / 60.0
    g["gap_prev_1"] = g["gap_min"].shift(1)
    g["gap_prev_2"] = g["gap_min"].shift(2)

    g["gap_mean_5"] = g["gap_min"].rolling(5, min_periods=2).mean()
    g["gap_std_5"]  = g["gap_min"].rolling(5, min_periods=2).std()
    g["gap_max_5"]  = g["gap_min"].rolling(5, min_periods=2).max()
    g["gap_min_5"]  = g["gap_min"].rolling(5, min_periods=2).min()

    g["gap_ratio"]  = g["gap_min"] / (g["gap_prev_1"] + 1e-6)
    g["gap_accel"]  = g["gap_min"] - g["gap_prev_1"]
    g["gap_trend_5"] = (
        g["gap_min"].rolling(5, min_periods=3)
        .apply(slope_from_series, raw=False)
    )

    # Enrichissements gap
    g["gap_ratio_vs_mean"] = g["gap_min"] / (g["gap_mean_5"] + 1e-6)
    g["gap_ratio_vs_max"]  = g["gap_min"] / (g["gap_max_5"]  + 1e-6)

    # Est-ce le plus long gap depuis le début de l'épisode ?
    g["is_longest_gap_so_far"] = (
        g["gap_min"] >= g["gap_min"].expanding().max()
    ).astype(float)

    # Combien de fois le gap a doublé sur les 3 derniers ?
    g["gap_doubling_count"] = (
        (g["gap_min"]    > 2 * g["gap_prev_1"] + 1e-6).astype(float) +
        (g["gap_prev_1"] > 2 * g["gap_prev_2"] + 1e-6).astype(float)
    )

    # Tendance exponentielle (log-gap)
    g["log_gap"] = np.log1p(g["gap_min"].clip(lower=0))
    g["log_gap_trend_5"] = (
        g["log_gap"].rolling(5, min_periods=3)
        .apply(slope_from_series, raw=False)
    )

    # ----------------------------------------------------------
    # GROUPE 2 — ACTIVITÉ (comptages sur fenêtres temporelles)
    # ----------------------------------------------------------
    t = g.set_index("date")

    g["strikes_5m"]  = t["inside20"].rolling("5min").count().values
    g["strikes_10m"] = t["inside20"].rolling("10min").count().values
    g["strikes_30m"] = t["inside20"].rolling("30min").count().values

    g["inside20_10m"] = t["inside20"].rolling("10min").sum().values
    g["inside20_30m"] = t["inside20"].rolling("30min").sum().values

    ring20_40 = ((g["dist"] > 20)  & (g["dist"] <= 40)).astype(float)
    ring40_80 = ((g["dist"] > 40)  & (g["dist"] <= 80)).astype(float)
    t_ring = pd.DataFrame(
        {"r20_40": ring20_40.values, "r40_80": ring40_80.values},
        index=g["date"]
    )
    g["ring20_40_10m"] = t_ring["r20_40"].rolling("10min").sum().values
    g["ring40_80_10m"] = t_ring["r40_80"].rolling("10min").sum().values

    # ----------------------------------------------------------
    # GROUPE 3 — SPATIAL (position et mouvement de l'orage)
    # ----------------------------------------------------------
    g["last_distance"]    = g["dist"]
    g["mean_distance_5"]  = g["dist"].rolling(5,  min_periods=2).mean()
    g["mean_distance_10"] = g["dist"].rolling(10, min_periods=3).mean()
    g["min_distance_5"]   = g["dist"].rolling(5,  min_periods=2).min()
    g["distance_slope_5"] = (
        g["dist"].rolling(5, min_periods=3)
        .apply(slope_from_series, raw=False)
    )
    g["distance_delta"]   = g["dist"].diff()
    g["radial_speed"]     = g["distance_delta"] / (g["gap_min"] + 1e-6)
    g["mean_radial_speed_5"] = g["radial_speed"].rolling(5, min_periods=2).mean()

    # Signal combiné : s'éloigne ET ralentit
    g["departing_and_slowing"] = (
        (g["distance_slope_5"] > 0) &
        (g["gap_accel"] > 0)
    ).astype(float)

    # Orage loin récemment (rien à moins de 15 km sur 5 derniers)
    g["closest_recent_far"] = (g["min_distance_5"] > 15).astype(float)

    # Dispersion azimutale (front organisé vs orage qui se désagrège)
    g["azimuth_std_5"]  = (
        g["azimuth"].rolling(5,  min_periods=3)
        .apply(circular_std_deg, raw=False)
    )
    g["azimuth_std_10"] = (
        g["azimuth"].rolling(10, min_periods=3)
        .apply(circular_std_deg, raw=False)
    )

    # Temps depuis le dernier éclair DANS la zone des 20 km
    inside_dates = g["date"].where(g["inside20"] == 1)
    g["last_inside20_date"] = inside_dates.ffill()
    g["time_since_last_inside20_min"] = (
        (g["date"] - g["last_inside20_date"]).dt.total_seconds() / 60.0
    )

    # ----------------------------------------------------------
    # GROUPE 4 — INTRA-NUAGE (chimie de l'orage)
    # ----------------------------------------------------------
    if "icloud" in g.columns:
        t_ic = g.set_index("date")["icloud"]

        g["icloud_frac_5m"]  = t_ic.rolling("5min").mean().values
        g["icloud_frac_10m"] = t_ic.rolling("10min").mean().values
        g["icloud_frac_20m"] = t_ic.rolling("20min").mean().values

        # Tendance : est-ce que la proportion IC monte ?
        g["icloud_frac_trend"] = g["icloud_frac_5m"] - g["icloud_frac_20m"]

        # Dernier éclair était intra-nuage ?
        g["last_was_icloud"] = g["icloud"]

        # Nombre d'éclairs sol dans les 10 derniers
        g["n_sol_last_10"] = (
            (g["icloud"] == 0).rolling(10, min_periods=1).sum()
        )

        # Les 5 derniers éclairs sont-ils tous intra-nuage ?
        g["last_5_all_icloud"] = (
            g["icloud"].rolling(5, min_periods=5).sum() == 5
        ).astype(float)
    else:
        for col in ["icloud_frac_5m", "icloud_frac_10m", "icloud_frac_20m",
                    "icloud_frac_trend", "last_was_icloud",
                    "n_sol_last_10", "last_5_all_icloud"]:
            g[col] = np.nan

    # ----------------------------------------------------------
    # GROUPE 5 — AMPLITUDE (intensité de l'orage)
    # ----------------------------------------------------------
    g["abs_amplitude"]    = g["amplitude"].abs()
    g["mean_abs_amp_5"]   = g["abs_amplitude"].rolling(5,  min_periods=2).mean()
    g["mean_abs_amp_10"]  = g["abs_amplitude"].rolling(10, min_periods=2).mean()
    g["max_abs_amp_10"]   = g["abs_amplitude"].rolling(10, min_periods=2).max()
    g["std_amp_5"]        = g["abs_amplitude"].rolling(5,  min_periods=2).std()
    g["amp_slope_5"]      = (
        g["abs_amplitude"].rolling(5, min_periods=3)
        .apply(slope_from_series, raw=False)
    )
    g["amp_slope_10"]     = (
        g["abs_amplitude"].rolling(10, min_periods=3)
        .apply(slope_from_series, raw=False)
    )

    # Ratio amplitude actuelle / max historique de l'alerte
    g["amp_vs_storm_max"] = (
        g["abs_amplitude"] /
        (g["abs_amplitude"].expanding().max() + 1e-6)
    )

    # Signal combiné : faible intensité ET gap croissant
    g["weak_and_slowing"] = (
        (g["amp_vs_storm_max"] < 0.2) &
        (g["gap_accel"] > 0)
    ).astype(float)

    # ----------------------------------------------------------
    # GROUPE 6 — MÉMOIRE LONGUE (contexte global de l'alerte)
    # ----------------------------------------------------------
    # Épisode = séquence continue (gap > 60 min = nouvel épisode)
    new_episode = g["gap_min"].isna() | (g["gap_min"] > 60)
    g["episode_id"] = new_episode.cumsum()

    episode_start = g.groupby("episode_id")["date"].transform("min")
    g["storm_age_min"] = (
        (g["date"] - episode_start).dt.total_seconds() / 60.0
    )
    g["storm_age_log"] = np.log1p(g["storm_age_min"])

    # Taux moyen d'éclairs depuis le début de l'épisode (éclairs/min)
    g["cumcount_in_episode"] = (
        g.groupby("episode_id").cumcount() + 1
    ).astype(float)
    g["mean_rate_all"] = g["cumcount_in_episode"] / (g["storm_age_min"] + 1e-6)

    # Taux actuel vs taux moyen
    g["current_rate"] = 1.0 / (g["gap_min"] + 1e-6)
    g["rate_vs_mean"] = g["current_rate"] / (g["mean_rate_all"] + 1e-6)

    # Éclairs sol cumulés dans l'épisode
    g["cumul_sol_in_episode"] = (
        (g["icloud"] == 0)
        .groupby(g["episode_id"])
        .cumsum()
        .astype(float)
    ) if "icloud" in g.columns else np.nan

    # ----------------------------------------------------------
    # GROUPE 7 — FEATURES D'INTERACTION
    # ----------------------------------------------------------
    # Score de fin d'orage : somme de 5 signaux binaires indépendants
    g["end_score"] = (
        (g["gap_min"] > 10).astype(float) +
        (g["distance_slope_5"] > 0).astype(float) +
        (g.get("icloud_frac_5m", pd.Series(0, index=g.index)) > 0.8).astype(float) +
        (g["amp_vs_storm_max"] < 0.3).astype(float) +
        (g["inside20_10m"] == 0).astype(float)
    )

    # Gap × distance : long silence + orage loin
    g["gap_x_dist"] = g["gap_min"] * g["last_distance"]

    # Gap × intra-nuage
    g["gap_x_icloud_frac"] = (
        g["gap_min"] * g.get("icloud_frac_5m", pd.Series(0, index=g.index))
    )

    # ----------------------------------------------------------
    # CONTEXTE TEMPOREL
    # ----------------------------------------------------------
    g["hour"]  = g["date"].dt.hour
    g["month"] = g["date"].dt.month

    return g


print("Construction des features...")
df = (
    df.groupby("airport", group_keys=False)
    .apply(add_features)
    .reset_index(drop=True)
)
print("Features construites.")

# ============================================================
# 4. TARGET
# y = 1 → pas d'éclair nuage-sol dans les HORIZON_MIN minutes
# y = 0 → au moins un éclair dangereux dans les HORIZON_MIN minutes
# ============================================================
def add_target(g, horizon_min=HORIZON_MIN):
    g = g.sort_values("date").copy()
    times  = g["date"].values
    inside = g["inside20"].values

    future = np.zeros(len(g), dtype=int)
    for i in range(len(g)):
        t0    = times[i]
        upper = t0 + np.timedelta64(horizon_min, "m")
        mask  = (times > t0) & (times <= upper) & (inside == 1)
        future[i] = int(mask.any())

    g["target"] = 1 - future   # 1 = sûr de lever, 0 = danger
    return g


print("Construction de la target...")
df = (
    df.groupby("airport", group_keys=False)
    .apply(add_target, horizon_min=HORIZON_MIN)
    .reset_index(drop=True)
)
print(f"Target construite. Taux sûr (y=1) : {df['target'].mean():.3f}")

# ============================================================
# 5. LISTE DES FEATURES
# ============================================================
FEATURES = [
    # Gaps
    "gap_min", "gap_prev_1", "gap_prev_2",
    "gap_mean_5", "gap_std_5", "gap_max_5",
    "gap_ratio", "gap_accel", "gap_trend_5",
    "gap_ratio_vs_mean", "gap_ratio_vs_max",
    "is_longest_gap_so_far", "gap_doubling_count",
    "log_gap_trend_5",

    # Activité
    "strikes_5m", "strikes_10m", "strikes_30m",
    "inside20_10m", "inside20_30m",
    "ring20_40_10m", "ring40_80_10m",

    # Spatial
    "last_distance", "mean_distance_5", "mean_distance_10",
    "min_distance_5", "distance_slope_5",
    "radial_speed", "mean_radial_speed_5",
    "departing_and_slowing", "closest_recent_far",
    "azimuth_std_5", "azimuth_std_10",
    "time_since_last_inside20_min",

    # Intra-nuage
    "icloud_frac_5m", "icloud_frac_10m", "icloud_frac_20m",
    "icloud_frac_trend", "last_was_icloud",
    "n_sol_last_10", "last_5_all_icloud",

    # Amplitude
    "mean_abs_amp_5", "mean_abs_amp_10", "max_abs_amp_10",
    "std_amp_5", "amp_slope_5", "amp_slope_10",
    "amp_vs_storm_max", "weak_and_slowing",

    # Mémoire longue
    "storm_age_min", "storm_age_log",
    "mean_rate_all", "rate_vs_mean",
    "cumul_sol_in_episode",

    # Interactions
    "end_score", "gap_x_dist", "gap_x_icloud_frac",

    # Temporel
    "hour", "month",
]

TARGET = "target"

# ============================================================
# 6. SPLIT TRAIN / TEST STRATIFIÉ PAR AÉROPORT
# ============================================================
print("\nSplit train/test stratifié par aéroport...")

train_ids, test_ids = [], []

for airport, group in df.groupby("airport"):
    # Trier par date et prendre les 80% plus anciens pour le train
    dates_sorted = group["date"].sort_values()
    cutoff = dates_sorted.iloc[int(len(dates_sorted) * TRAIN_RATIO)]

    train_ids.extend(group[group["date"] < cutoff].index.tolist())
    test_ids.extend(group[group["date"] >= cutoff].index.tolist())

train_df = df.loc[train_ids].copy()
valid_df  = df.loc[test_ids].copy()

# Supprimer les lignes avec target ou features manquantes
train_df = train_df.dropna(subset=[TARGET]).copy()
valid_df  = valid_df.dropna(subset=[TARGET]).copy()

# Garder uniquement les features disponibles dans le DataFrame
FEATURES = [f for f in FEATURES if f in df.columns]

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].astype(int)
X_valid = valid_df[FEATURES].copy()
y_valid  = valid_df[TARGET].astype(int)

print(f"Train : {len(X_train)} lignes | taux sûr : {y_train.mean():.3f}")
print(f"Test  : {len(X_valid)} lignes  | taux sûr : {y_valid.mean():.3f}")
print(f"Nombre de features : {len(FEATURES)}")

# ============================================================
# 7. ENTRAÎNEMENT
# ============================================================
ratio_desequilibre = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"\nRatio déséquilibre : {ratio_desequilibre:.2f}")

model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=40,
    max_depth=6,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=ratio_desequilibre,
    n_jobs=2,
    random_state=42,
    verbose=-1,
)

print("Entraînement...")
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="binary_logloss",
)

# Calibration des probabilités
print("Calibration des probabilités...")
model_calibrated = CalibratedClassifierCV(
    estimator=model,
    method="isotonic",
    cv="prefit"
)
model_calibrated.fit(X_valid, y_valid)
print("Modèle prêt.")

# ============================================================
# 8. ÉVALUATION STANDARD
# ============================================================
valid_proba = model_calibrated.predict_proba(X_valid)[:, 1]
valid_pred  = (valid_proba >= 0.5).astype(int)

auc   = roc_auc_score(y_valid, valid_proba)
ap    = average_precision_score(y_valid, valid_proba)
brier = brier_score_loss(y_valid, valid_proba)
cm    = confusion_matrix(y_valid, valid_pred)
tn, fp, fn, tp = cm.ravel()

print("\n=== MÉTRIQUES STANDARD ===")
print(f"AUC-ROC      : {auc:.4f}")
print(f"PR-AUC       : {ap:.4f}")
print(f"Brier Score  : {brier:.4f}")
print(f"Accuracy     : {accuracy_score(y_valid, valid_pred):.4f}")
print(f"F1-score     : {f1_score(y_valid, valid_pred):.4f}")
print(f"\nMatrice de confusion [[TN, FP], [FN, TP]] :")
print(cm)

false_clear_rate = fp / (fp + tp) if (fp + tp) > 0 else np.nan
missed_stop_rate = fn / (fn + tn) if (fn + tn) > 0 else np.nan
print(f"\n=== MÉTRIQUES OPÉRATIONNELLES ===")
print(f"Taux de fausse levée (dangereux) : {false_clear_rate:.4f}")
print(f"Taux d'alertes non levées        : {missed_stop_rate:.4f}")

# ============================================================
# 9. ANALYSE PAR SEUIL
# ============================================================
print("\n=== ANALYSE PAR SEUIL ===")
thresholds = np.arange(0.50, 0.99, 0.05)
rows = []
for th in thresholds:
    pred = (valid_proba >= th).astype(int)
    if len(np.unique(pred)) < 2:
        continue
    cm_th = confusion_matrix(y_valid, pred)
    tn_t, fp_t, fn_t, tp_t = cm_th.ravel()
    false_clear = fp_t / (fp_t + tp_t) if (fp_t + tp_t) > 0 else np.nan
    rows.append({
        "seuil":            round(th, 2),
        "taux_fausse_levee": round(false_clear, 4),
        "precision":        round(precision_score(y_valid, pred, zero_division=0), 4),
        "recall":           round(recall_score(y_valid, pred, zero_division=0), 4),
        "stop_rate":        round(pred.mean(), 4),
    })

threshold_df = pd.DataFrame(rows)
print(threshold_df.to_string(index=False))

# ============================================================
# 10. IMPORTANCE DES FEATURES
# ============================================================
feat_imp = (
    pd.DataFrame({
        "feature":    FEATURES,
        "importance": model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\n=== TOP 20 FEATURES ===")
print(feat_imp.head(20).to_string(index=False))

# ============================================================
# 11. ÉVALUATION MÉTIER (gain de temps vs règle des 30 min)
# ============================================================
print("\n=== ÉVALUATION MÉTIER ===")

# On attache les probas au DataFrame de validation
valid_df = valid_df.copy()
valid_df["proba_safe"] = valid_proba   # P(sûr de lever l'alerte)

# Pour chaque alerte et chaque seuil, on simule la décision opérationnelle
# L'alerte est levée au PREMIER éclair où proba_safe >= seuil
# ET où un silence minimal de 5 min a été observé

def evaluate_metier(valid_df, df_raw, seuils, horizon_min=HORIZON_MIN):
    results = []

    for seuil in seuils:
        fausses_levees = 0
        vrais_gains    = []
        sans_decision  = 0

        # On groupe par épisode × aéroport
        for (airport, ep_id), group in valid_df.groupby(["airport", "episode_id"]):
            group = group.sort_values("date")

            # Dernière date réglementaire = dernier éclair + 30 min
            t_last_real     = group["date"].max()
            t_reglementaire = t_last_real + pd.Timedelta(minutes=horizon_min)

            # Candidats : proba_safe >= seuil ET gap >= 5 min
            candidats = group[
                (group["proba_safe"] >= seuil) &
                (group["gap_min"] >= 5)
            ]

            if candidats.empty:
                sans_decision += 1
                continue

            t_decision = candidats["date"].iloc[0]

            # Vérification sécurité : y a-t-il un éclair sol dans la zone après ?
            eclairs_post = df_raw[
                (df_raw["airport"] == airport) &
                (df_raw["date"] > t_decision) &
                (df_raw["inside20"] == 1) &
                (df_raw["icloud"] == 0)
            ]

            if len(eclairs_post) > 0:
                fausses_levees += 1
            else:
                gain = (t_reglementaire - t_decision).total_seconds() / 60.0
                if gain > 0:
                    vrais_gains.append(gain)

        n_episodes  = valid_df.groupby(["airport", "episode_id"]).ngroups
        n_decisions = n_episodes - sans_decision

        results.append({
            "seuil":              seuil,
            "taux_fausse_levee":  fausses_levees / max(n_decisions, 1),
            "gain_moyen_min":     round(np.mean(vrais_gains), 2) if vrais_gains else 0.0,
            "gain_median_min":    round(np.median(vrais_gains), 2) if vrais_gains else 0.0,
            "n_fausses_levees":   fausses_levees,
            "n_episodes":         n_episodes,
            "couverture":         round(n_decisions / max(n_episodes, 1), 3),
        })

    return pd.DataFrame(results)


SEUILS_METIER = [0.70, 0.75, 0.80, 0.85, 0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99]
df_metier = evaluate_metier(valid_df, df, SEUILS_METIER)

print(df_metier.to_string(index=False))

# Seuil optimal : taux fausse levée < 1%
optimal = df_metier[df_metier["taux_fausse_levee"] <= 0.01]
print("\n=== RAPPORT FINAL ===")
print(f"AUC-ROC : {auc:.4f}")
print(f"Brier   : {brier:.4f}")
if len(optimal) > 0:
    best = optimal.iloc[0]
    print(f"\nSeuil recommandé    : {best['seuil']:.2f}")
    print(f"Gain moyen garanti  : {best['gain_moyen_min']:.1f} min/alerte")
    print(f"Taux fausse levée   : {best['taux_fausse_levee']*100:.2f}%")
    print(f"Couverture alertes  : {best['couverture']*100:.1f}%")
else:
    print("\nAucun seuil ne satisfait taux fausse levée < 1%.")
    print("→ Améliorer le modèle ou accepter un seuil plus permissif.")

# ============================================================
# 12. SAUVEGARDE
# ============================================================
import joblib
joblib.dump(model_calibrated, "./model_calibrated_v2.pkl")
joblib.dump(FEATURES,         "./feature_cols_v2.pkl")
print("\nModèle sauvegardé → model_calibrated_v2.pkl")

Dataset chargé : 507071 lignes, 5 aéroports
Période : 2016-01-02 01:10:41+00:00 → 2022-12-21 11:20:11+00:00
Construction des features...
Features construites.
Construction de la target...
Target construite. Taux sûr (y=1) : 0.049

Split train/test stratifié par aéroport...
Train : 405654 lignes | taux sûr : 0.051
Test  : 101417 lignes  | taux sûr : 0.041
Nombre de features : 58

Ratio déséquilibre : 18.53
Entraînement...
Calibration des probabilités...
Modèle prêt.

=== MÉTRIQUES STANDARD ===
AUC-ROC      : 0.9555
PR-AUC       : 0.4870
Brier Score  : 0.0267
Accuracy     : 0.9632
F1-score     : 0.4026

Matrice de confusion [[TN, FP], [FN, TP]] :
[[96429   816]
 [ 2915  1257]]

=== MÉTRIQUES OPÉRATIONNELLES ===
Taux de fausse levée (dangereux) : 0.3936
Taux d'alertes non levées        : 0.0293

=== ANALYSE PAR SEUIL ===
 seuil  taux_fausse_levee  precision  recall  stop_rate
  0.50             0.3936     0.6064  0.3013     0.0204
  0.55             0.3456     0.6544  0.2011     0.0126
  